# RVC — Pipeline Đầu Cuối: Chuẩn Bị → Training → Inference → Đánh Giá

Notebook tích hợp toàn bộ 3 giai đoạn vào một luồng chạy liên tục.

## Pipeline
```
[LibriTTS] → Chuẩn bị data → Preprocess → Trích F0 + Hubert → FAISS Index
                                                        ↓
              [Vòng lặp — chạy nhiều lần nếu COMPARE_MULTIPLE_MODELS=True]
              ┌────────────────────────────────────────────────────────┐
              │  Train đến epoch_target (continue từ checkpoint trước) │
              │  → Export model_<N>ep.pth                              │
              │  → Inference source_test → converted_output/model_<N>/ │
              │  → Đánh giá: Speaker Similarity + MOS + WER/CER        │
              └────────────────────────────────────────────────────────┘
                                                        ↓
              Bảng tổng hợp + Biểu đồ so sánh các model
```

## Cách dùng
1. Sửa **Cell 2 (CẤU HÌNH)** — đặt cờ và tham số
2. Chạy tuần tự từ Cell 3 đến hết
3. Cell 10 (Vòng lặp chính) sẽ tự động train → export → infer → evaluate cho từng mốc epoch

## Yêu cầu
- Working directory: thư mục `rvc_standalone` (có `infer/`, `configs/`, `training_pipeline/`)
- Dataset LibriTTS đã tải (`train-clean-100` và `test-clean`)
- `python tools/download_assets.py` đã chạy để tải weights

In [1]:
# ================================================================
# CẤU HÌNH — Đọc và sửa phần này trước khi chạy bất cứ điều gì
# ================================================================
from pathlib import Path

# ---------------------------------------------------------------
# CỜ CHÍNH
# ---------------------------------------------------------------
# True  = So sánh nhiều model với số epoch khác nhau
# False = Chỉ train 1 model duy nhất
COMPARE_MULTIPLE_MODELS = True

# Danh sách số epoch mục tiêu — CHỈ DÙNG khi COMPARE_MULTIPLE_MODELS = True
# Training sẽ CONTINUE từ checkpoint, không train lại từ đầu.
# Ví dụ [50, 100, 200]: train 50ep → tiếp tục đến 100ep → tiếp tục đến 200ep
EPOCH_LIST = [50, 100, 200]

# Số epoch — CHỈ DÙNG khi COMPARE_MULTIPLE_MODELS = False
SINGLE_MODEL_EPOCHS = 100

# ---------------------------------------------------------------
# DATASET
# ---------------------------------------------------------------
DATASET_ROOT         = Path('./rvc_eval_project/raw_data/LibriTTS')
ARCHIVE_DIR          = Path('./downloads')
TARGET_SPLIT         = 'train-clean-100'
SOURCE_SPLIT         = 'test-clean'
TARGET_SPEAKER_ID    = None   # None = tự chọn speaker có đủ data
SOURCE_SPEAKER_ID    = None
TARGET_TRAIN_SECONDS = 10 * 60  # 10 phút audio để train RVC
TARGET_EVAL_SECONDS  = 2  * 60  # 2 phút giữ lại để đánh giá similarity
SOURCE_TEST_COUNT    = 10       # Số câu source test
MIN_SOURCE_DURATION  = 2.0
MAX_SOURCE_DURATION  = 12.0
PREPARED_AUDIO_SR    = None     # None = giữ SR gốc

# Bật True nếu muốn tạo lại thư mục prepared_data từ đầu
RESET_PREPARED_DATA  = False

# ---------------------------------------------------------------
# TRAINING
# ---------------------------------------------------------------
EXPERIMENT_NAME    = 'rvc_experiment'  # Tên thư mục logs/<tên>/
SAMPLE_RATE_LABEL  = '40k'             # '32k' | '40k' | '48k'
RVC_VERSION        = 'v2'              # 'v1' | 'v2' — luôn dùng v2
IF_F0              = True              # True = giữ cao độ (cho hát)
F0_METHOD_TRAIN    = 'rmvpe'           # 'rmvpe' | 'harvest' | 'pm' | 'crepe'
NUM_PROCESSES      = 4                 # CPU cores cho preprocess
GPU_DEVICES        = '0'              # GPU ID: '0' | '0-1'
BATCH_SIZE         = 4                 # RTX 3050/4GB: 4; GPU mạnh: 8-16
SAVE_EVERY_EPOCH   = 10                # Lưu checkpoint mỗi N epoch
SKIP_INDEX         = False             # False = tạo FAISS index

# ---------------------------------------------------------------
# INFERENCE
# ---------------------------------------------------------------
INFER_SPEAKER_ID    = 0       # 0 cho model 1 giọng
INFER_F0_UP_KEY     = 0       # 0 = giữ tông; +N/-N = dịch N nửa cung
INFER_F0_METHOD     = 'rmvpe' # Khuyến nghị: 'rmvpe'
INFER_INDEX_RATE    = 0.75    # 0.0-1.0; 0.0 = không dùng index
INFER_FILTER_RADIUS = 3       # 0-7; chỉ ảnh hưởng khi dùng 'harvest'
INFER_RESAMPLE_SR   = 0       # 0 = giữ SR model; 44100/48000 nếu cần
INFER_RMS_MIX_RATE  = 0.25    # 0.0-1.0
INFER_PROTECT       = 0.33    # 0.0-0.5

# ---------------------------------------------------------------
# ĐÁNH GIÁ
# ---------------------------------------------------------------
ASR_MODEL_ID      = 'openai/whisper-small'
ASR_LANGUAGE      = 'english'  # 'english' cho LibriTTS; None = tự phát hiện
RUN_PREDICTED_MOS = True
MOS_BACKEND       = 'utmos_torchhub'

# ---------------------------------------------------------------
# ĐƯỜNG DẪN DỰ ÁN
# ---------------------------------------------------------------
PROJECT_DIR    = Path('./rvc_eval_project')
PREPARED_DIR   = PROJECT_DIR / 'prepared_data'
CONVERTED_ROOT = PROJECT_DIR / 'converted_output'
RESULTS_DIR    = PROJECT_DIR / 'results'

for _d in [PROJECT_DIR, PREPARED_DIR, CONVERTED_ROOT, RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# Tổng hợp danh sách epoch sẽ chạy
EPOCH_LIST_EFFECTIVE = sorted(EPOCH_LIST) if COMPARE_MULTIPLE_MODELS else [SINGLE_MODEL_EPOCHS]

print('=' * 55)
print('CẤU HÌNH')
print('=' * 55)
print(f'  So sánh nhiều model : {COMPARE_MULTIPLE_MODELS}')
print(f'  Danh sách epoch     : {EPOCH_LIST_EFFECTIVE}')
print(f'  Số model sẽ train   : {len(EPOCH_LIST_EFFECTIVE)}')
print(f'  Experiment          : {EXPERIMENT_NAME}')
print(f'  Sample rate / Ver   : {SAMPLE_RATE_LABEL} / {RVC_VERSION}')
print(f'  Batch size / GPU    : {BATCH_SIZE} / {GPU_DEVICES}')
print(f'  Save mỗi N epoch    : {SAVE_EVERY_EPOCH}')
print(f'  FAISS index         : {"BỎ QUA" if SKIP_INDEX else "SẼ TẠO"}')
print('=' * 55)

CẤU HÌNH
  So sánh nhiều model : True
  Danh sách epoch     : [50, 100, 200]
  Số model sẽ train   : 3
  Experiment          : rvc_experiment
  Sample rate / Ver   : 40k / v2
  Batch size / GPU    : 4 / 0
  Save mỗi N epoch    : 10
  FAISS index         : SẼ TẠO


In [5]:
# ================================================================
# KIỂM TRA MÔI TRƯỜNG
# ================================================================
import subprocess
import pathlib
import torch

CWD = pathlib.Path.cwd().resolve()
print('Working directory:', CWD)

# Cấu trúc thư mục
required_dirs = ['infer', 'configs', 'training_pipeline', 'assets', 'logs']
missing_dirs = [d for d in required_dirs if not (CWD / d).is_dir()]
if missing_dirs:
    print('\n❌ Thiếu thư mục:', missing_dirs)
    print('   Mở Jupyter từ thư mục rvc_standalone.')
else:
    print('✅ Cấu trúc thư mục OK')

# GPU
print(f'\nPyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} | {props.total_memory/1024**3:.1f} GB VRAM')
    print('✅ GPU sẵn sàng')
else:
    print('⚠️  Không có GPU — training và inference sẽ rất chậm')

# Weights
weights_to_check = [
    ('assets/hubert/hubert_base.pt',      'Hubert (bắt buộc)'),
    ('assets/rmvpe/rmvpe.pt',             'RMVPE (cần nếu dùng rmvpe)'),
    ('assets/pretrained_v2/f0G40k.pth',   'Pretrained G 40k'),
    ('assets/pretrained_v2/f0D40k.pth',   'Pretrained D 40k'),
]
print('\nModel weights:')
all_weights_ok = True
for rel_path, desc in weights_to_check:
    f = CWD / rel_path
    if f.exists():
        print(f'  ✓ {desc} ({f.stat().st_size/1024**2:.0f} MB)')
    else:
        print(f'  ✗ {desc} — THIẾU: {rel_path}')
        all_weights_ok = False
if not all_weights_ok:
    print('\n  → Chạy: python tools/download_assets.py')

# FFmpeg
try:
    r = subprocess.run(['ffmpeg', '-version'], capture_output=True, timeout=5)
    first_line = r.stdout.decode(errors='ignore').splitlines()[0] if r.stdout else ''
    print(f'\n✅ FFmpeg: {first_line[:60]}')
except FileNotFoundError:
    print('\n❌ FFmpeg không tìm thấy — cài FFmpeg và thêm vào PATH')

Working directory: /home/ngan-gcp-rvc/compare_rvc
✅ Cấu trúc thư mục OK

PyTorch  : 2.5.1+cu121
CUDA     : True
  GPU 0: Tesla P4 | 7.4 GB VRAM
✅ GPU sẵn sàng

Model weights:
  ✓ Hubert (bắt buộc) (181 MB)
  ✓ RMVPE (cần nếu dùng rmvpe) (173 MB)
  ✓ Pretrained G 40k (70 MB)
  ✓ Pretrained D 40k (136 MB)

✅ FFmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-202


In [4]:
# ================================================================
# CÀI THƯ VIỆN (chỉ cài những gì chưa có)
# ================================================================
import importlib.util
import subprocess
import sys

packages = {
    'numpy':       'numpy',
    'pandas':      'pandas',
    'tqdm':        'tqdm',
    'soundfile':   'soundfile',
    'librosa':     'librosa',
    'sklearn':     'scikit-learn',
    'matplotlib':  'matplotlib',
    'jiwer':       'jiwer',
    'transformers':'transformers',
    'speechbrain': 'speechbrain',
}

for import_name, pip_name in packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
    else:
        print(f'OK: {import_name}')

print('\n✅ Thư viện sẵn sàng')

OK: numpy
OK: pandas
OK: tqdm
OK: soundfile
OK: librosa
OK: sklearn
OK: matplotlib
Installing jiwer...


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Installing transformers...


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Installing speechbrain...


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip



✅ Thư viện sẵn sàng


In [6]:
# ================================================================
# IMPORT TẤT CẢ + THIẾT LẬP MÔI TRƯỜNG
# ================================================================
import os
import re
import sys
import time
import math
import json
import random
import shutil
import tarfile
import unicodedata
import pathlib
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import soundfile as sf
import librosa
import torch
import torchaudio
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import jiwer

# Setup working directory và sys.path
STANDALONE_ROOT = Path.cwd().resolve()
os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

# Env vars cho RVC inference
os.environ.setdefault('weight_root',       'assets/weights')
os.environ.setdefault('index_root',        'logs')
os.environ.setdefault('outside_index_root','assets/indices')
os.environ.setdefault('rmvpe_root',        'assets/rmvpe')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('STANDALONE_ROOT:', STANDALONE_ROOT)
print('DEVICE         :', DEVICE)
print('✅ Import xong')

/home/ngan-gcp-rvc/compare_rvc/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STANDALONE_ROOT: /home/ngan-gcp-rvc/compare_rvc
DEVICE         : cuda
✅ Import xong


In [7]:
# ================================================================
# KIỂM TRA LIBRI TTS VÀ QUÉT METADATA
# ================================================================
AUDIO_EXTS = {'.wav', '.flac', '.mp3', '.ogg', '.m4a'}


def find_split_dir(dataset_root: Path, split_name: str) -> Optional[Path]:
    candidates = [
        dataset_root / split_name,
        dataset_root / 'LibriTTS' / split_name,
        dataset_root.parent / 'LibriTTS' / split_name,
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    if dataset_root.exists():
        for p in dataset_root.rglob(split_name):
            if p.is_dir():
                return p
    return None


def read_audio_duration(path: Path) -> float:
    try:
        info = sf.info(str(path))
        return float(info.frames) / float(info.samplerate)
    except Exception:
        try:
            info = torchaudio.info(str(path))
            return float(info.num_frames) / float(info.sample_rate)
        except Exception:
            return np.nan


def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8').strip()
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1').strip()


def get_transcript_for_audio(audio_path: Path) -> str:
    for suffix in ['.normalized.txt', '.original.txt', '.txt']:
        cand = audio_path.with_suffix(suffix)
        if cand.exists():
            txt = read_text_file(cand)
            if txt:
                return txt
    utt_id = audio_path.stem
    for trans_file in audio_path.parent.glob('*.trans.txt'):
        try:
            for line in read_text_file(trans_file).splitlines():
                if line.startswith(utt_id + ' '):
                    return line[len(utt_id) + 1:].strip()
        except Exception:
            pass
    return ''


def infer_speaker_id(audio_path: Path, split_dir: Path) -> str:
    try:
        rel = audio_path.relative_to(split_dir)
        if len(rel.parts) >= 3:
            return str(rel.parts[0])
    except Exception:
        pass
    return audio_path.parent.name


def scan_split(split_name: str) -> pd.DataFrame:
    split_dir = find_split_dir(DATASET_ROOT, split_name)
    if split_dir is None:
        raise FileNotFoundError(f'Không tìm thấy split {split_name}.')
    audio_files = [p for p in split_dir.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTS]
    rows = []
    for p in tqdm(audio_files, desc=f'Scanning {split_name}'):
        rows.append({
            'split':      split_name,
            'speaker_id': infer_speaker_id(p, split_dir),
            'path':       str(p),
            'file_name':  p.name,
            'stem':       p.stem,
            'duration_sec': read_audio_duration(p),
            'transcript': get_transcript_for_audio(p),
        })
    return pd.DataFrame(rows)


# Kiểm tra splits trước khi scan
for split in [TARGET_SPLIT, SOURCE_SPLIT]:
    split_dir = find_split_dir(DATASET_ROOT, split)
    status = 'FOUND ' + str(split_dir) if split_dir else 'MISSING'
    print(f'{split}: {status}')

# Scan metadata (có thể mất vài phút với train-clean-100)
print('\nBắt đầu scan metadata...')
train_df  = scan_split(TARGET_SPLIT)
source_df = scan_split(SOURCE_SPLIT)

print(f'\ntrain_df : {train_df.shape}')
print(f'source_df: {source_df.shape}')

train_df.to_csv(RESULTS_DIR / f'metadata_{TARGET_SPLIT}.csv',  index=False, encoding='utf-8-sig')
source_df.to_csv(RESULTS_DIR / f'metadata_{SOURCE_SPLIT}.csv', index=False, encoding='utf-8-sig')
print('✅ Scan xong')

train-clean-100: MISSING
test-clean: MISSING

Bắt đầu scan metadata...


FileNotFoundError: Không tìm thấy split train-clean-100.

In [ ]:
# ================================================================
# CHỌN SPEAKER VÀ CHUẨN BỊ DỮ LIỆU
# ================================================================
import warnings
warnings.filterwarnings('ignore')


def speaker_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.groupby('speaker_id')
        .agg(
            n_files=('path', 'count'),
            total_sec=('duration_sec', 'sum'),
            n_with_transcript=('transcript', lambda x: sum(bool(str(v).strip()) for v in x)),
        )
        .reset_index()
    )
    out['total_min'] = out['total_sec'] / 60.0
    return out.sort_values('total_sec', ascending=False)


def convert_to_wav(src: Path, dst: Path, sr: Optional[int] = None):
    dst.parent.mkdir(parents=True, exist_ok=True)
    y, orig_sr = librosa.load(str(src), sr=sr, mono=True)
    out_sr = sr if sr is not None else orig_sr
    if np.max(np.abs(y)) > 1.0:
        y = y / np.max(np.abs(y))
    sf.write(str(dst), y, out_sr)


def prepare_audio_set(df: pd.DataFrame, output_dir: Path, prefix: str, sr: Optional[int] = None) -> pd.DataFrame:
    rows = []
    output_dir.mkdir(parents=True, exist_ok=True)
    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f'Preparing {prefix}'), start=1):
        src = Path(row['path'])
        out_name = f'{prefix}_{idx:03d}.wav'
        dst = output_dir / out_name
        convert_to_wav(src, dst, sr=sr)
        rows.append({
            'id':            f'{prefix}_{idx:03d}',
            'file':          out_name,
            'path':          str(dst),
            'original_path': str(src),
            'speaker_id':    row['speaker_id'],
            'duration_sec':  read_audio_duration(dst),
            'transcript':    row.get('transcript', ''),
        })
    return pd.DataFrame(rows)


# Kiểm tra prepared_data có sẵn chưa
manifests_exist = all([
    (PREPARED_DIR / 'target_train_manifest.csv').exists(),
    (PREPARED_DIR / 'target_eval_manifest.csv').exists(),
    (PREPARED_DIR / 'source_test_manifest.csv').exists(),
])

if manifests_exist and not RESET_PREPARED_DATA:
    print('✅ Đã có prepared_data — bỏ qua chuẩn bị lại (đặt RESET_PREPARED_DATA=True để tạo lại)')
else:
    if RESET_PREPARED_DATA and PREPARED_DIR.exists():
        shutil.rmtree(PREPARED_DIR)
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)

    # Chọn speakers
    train_spk  = speaker_summary(train_df)
    source_spk = speaker_summary(source_df)

    needed = TARGET_TRAIN_SECONDS + TARGET_EVAL_SECONDS
    _target_id = TARGET_SPEAKER_ID
    if _target_id is None:
        cands = train_spk[train_spk['total_sec'] >= needed]
        if cands.empty:
            raise ValueError(f'Không có speaker đủ {needed/60:.1f} phút trong {TARGET_SPLIT}.')
        _target_id = str(cands.iloc[0]['speaker_id'])

    _source_id = SOURCE_SPEAKER_ID
    if _source_id is None:
        cands = source_spk[source_spk['n_with_transcript'] >= SOURCE_TEST_COUNT]
        cands2 = cands[cands['speaker_id'].astype(str) != str(_target_id)]
        if not cands2.empty:
            cands = cands2
        if cands.empty:
            raise ValueError(f'Không có source speaker đủ {SOURCE_TEST_COUNT} câu có transcript.')
        _source_id = str(cands.iloc[0]['speaker_id'])

    print(f'Target speaker: {_target_id}')
    print(f'Source speaker: {_source_id}')

    # Tách target thành train / eval
    target_all = (
        train_df[train_df['speaker_id'].astype(str) == str(_target_id)]
        .dropna(subset=['duration_sec'])
        .pipe(lambda d: d[d['duration_sec'] > 0])
        .sample(frac=1, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    target_train_rows, target_eval_rows = [], []
    used_idx, acc = set(), 0.0
    for i, row in target_all.iterrows():
        if acc < TARGET_TRAIN_SECONDS:
            target_train_rows.append(row); used_idx.add(i); acc += float(row['duration_sec'])
    acc_eval = 0.0
    for i, row in target_all.iterrows():
        if i not in used_idx and acc_eval < TARGET_EVAL_SECONDS:
            target_eval_rows.append(row); acc_eval += float(row['duration_sec'])

    target_train_sel = pd.DataFrame(target_train_rows)
    target_eval_sel  = pd.DataFrame(target_eval_rows)

    # Chọn source test
    source_pool = source_df[source_df['speaker_id'].astype(str) == str(_source_id)].copy()
    source_pool = source_pool[source_pool['transcript'].astype(str).str.len() > 0]
    source_pool = source_pool[source_pool['duration_sec'].between(MIN_SOURCE_DURATION, MAX_SOURCE_DURATION)]
    if len(source_pool) < SOURCE_TEST_COUNT:
        source_pool = source_df[source_df['speaker_id'].astype(str) == str(_source_id)].copy()
        source_pool = source_pool[source_pool['transcript'].astype(str).str.len() > 0]
    source_sel = source_pool.sample(n=min(SOURCE_TEST_COUNT, len(source_pool)), random_state=RANDOM_SEED).reset_index(drop=True)

    print(f'Target train: {target_train_sel["duration_sec"].sum()/60:.1f} phút ({len(target_train_sel)} file)')
    print(f'Target eval : {target_eval_sel["duration_sec"].sum()/60:.1f} phút ({len(target_eval_sel)} file)')
    print(f'Source test : {len(source_sel)} file')

    # Chuẩn bị file WAV
    target_train_prep = prepare_audio_set(target_train_sel, PREPARED_DIR / 'target_train', 'target_train', PREPARED_AUDIO_SR)
    target_eval_prep  = prepare_audio_set(target_eval_sel,  PREPARED_DIR / 'target_eval',  'target_eval',  PREPARED_AUDIO_SR)
    source_test_prep  = prepare_audio_set(source_sel,       PREPARED_DIR / 'source_test',  'source',       PREPARED_AUDIO_SR)

    # Lưu manifest
    target_train_prep.to_csv(PREPARED_DIR / 'target_train_manifest.csv', index=False, encoding='utf-8-sig')
    target_eval_prep.to_csv(PREPARED_DIR  / 'target_eval_manifest.csv',  index=False, encoding='utf-8-sig')
    source_test_prep.to_csv(PREPARED_DIR  / 'source_test_manifest.csv',  index=False, encoding='utf-8-sig')
    source_test_prep[['file', 'transcript']].to_csv(PREPARED_DIR / 'transcript.csv', index=False, encoding='utf-8-sig')

    print(f'\n✅ Đã chuẩn bị dữ liệu tại: {PREPARED_DIR.resolve()}')

In [ ]:
# ================================================================
# KHỞI TẠO TRAINING + PREPROCESS + TRÍCH F0 + HUBERT
# ================================================================
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(message)s')

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()

print('Training config:')
print(f'  Python  : {config.python_cmd}')
print(f'  Device  : {config.device}')
print(f'  FP16    : {config.is_half}')

# Tạo TrainingParams — total_epochs và infer_weight_name sẽ được ghi đè trong vòng lặp
p = TrainingParams(
    experiment_name   = EXPERIMENT_NAME,
    trainset_dir      = str(PREPARED_DIR / 'target_train'),
    sample_rate_label = SAMPLE_RATE_LABEL,
    version           = RVC_VERSION,
    if_f0             = IF_F0,
    f0_method         = F0_METHOD_TRAIN,
    num_processes     = NUM_PROCESSES,
    gpu_devices_train = GPU_DEVICES,
    total_epochs      = max(EPOCH_LIST_EFFECTIVE),  # placeholder — ghi đè trong loop
    save_every_epoch  = SAVE_EVERY_EPOCH,
    batch_size        = BATCH_SIZE,
    skip_index        = SKIP_INDEX,
    infer_weight_name = 'temp_model',               # placeholder — ghi đè trong loop
    extract_infer_pth = False,
    speaker_id        = 0,
)

# ---- PREPROCESS ----
exp_dir = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
gt_wavs_existing = list((exp_dir / '0_gt_wavs').glob('*.wav')) if (exp_dir / '0_gt_wavs').exists() else []

if gt_wavs_existing:
    print(f'\n✅ Preprocess đã làm trước ({len(gt_wavs_existing)} clips) — bỏ qua')
else:
    print('\nBước PREPROCESS...')
    train_steps.step_preprocess(STANDALONE_ROOT, config, p)
    gt_wavs_existing = list((exp_dir / '0_gt_wavs').glob('*.wav'))
    print(f'✅ Preprocess xong: {len(gt_wavs_existing)} clips')

# ---- TRÍCH F0 + HUBERT ----
feature_dir_name = '3_feature768' if RVC_VERSION == 'v2' else '3_feature256'
feature_dir = exp_dir / feature_dir_name
features_existing = list(feature_dir.glob('*.npy')) if feature_dir.exists() else []

if features_existing:
    print(f'\n✅ Features đã trích ({len(features_existing)} file) — bỏ qua')
else:
    print('\nBước TRÍCH F0 + HUBERT features...')
    train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)
    features_existing = list(feature_dir.glob('*.npy'))
    print(f'✅ Trích xong: {len(features_existing)} feature files')

# Kiểm tra tổng
print(f'\nKiểm tra:')
print(f'  gt_wavs   : {len(gt_wavs_existing)}')
print(f'  features  : {len(features_existing)}')
if len(features_existing) == len(gt_wavs_existing) and len(gt_wavs_existing) > 0:
    print('✅ Số files khớp — sẵn sàng train')
else:
    print('⚠️  Số files không khớp — kiểm tra log trong logs/' + EXPERIMENT_NAME)

In [ ]:
# ================================================================
# TẠO FAISS INDEX (1 lần — features không đổi giữa các model)
# ================================================================
exp_dir = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
existing_indexes = list(exp_dir.glob('added_*.index'))

if existing_indexes:
    print(f'✅ FAISS index đã có:')
    for f in existing_indexes:
        print(f'   {f.name}  ({f.stat().st_size/1024**2:.1f} MB)')
elif SKIP_INDEX:
    print('SKIP_INDEX=True — bỏ qua tạo FAISS index')
    print('Khi inference, INFER_INDEX_RATE sẽ không có tác dụng.')
else:
    print('Đang tạo FAISS Index...')
    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)
    existing_indexes = list(exp_dir.glob('added_*.index'))
    if existing_indexes:
        print(f'✅ Index tạo xong: {existing_indexes[-1].name}')
    else:
        print('⚠️  Không tìm thấy index file sau khi tạo')

In [ ]:
# ================================================================
# LOAD CÁC MODEL ĐÁNH GIÁ + TÍNH BASELINE EMBEDDING
# (Chạy 1 lần, dùng lại cho tất cả model trong vòng lặp)
# ================================================================

# ---- Speaker Embedding (SpeechBrain ECAPA-TDNN) ----
from speechbrain.inference.speaker import EncoderClassifier

SPEAKER_MODEL_DIR = str(PROJECT_DIR / 'pretrained_models' / 'spkrec-ecapa-voxceleb')
print('Load speaker embedding model...')
speaker_encoder = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir=SPEAKER_MODEL_DIR,
    run_opts={'device': DEVICE},
)
print('✅ Speaker encoder OK')


def load_audio_16k(path: Path) -> torch.Tensor:
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.squeeze(0)
    if wav.abs().max() > 0:
        wav = wav / wav.abs().max()
    return wav


def speaker_embedding(path: Path) -> np.ndarray:
    wav = load_audio_16k(path).to(DEVICE)
    with torch.no_grad():
        emb = speaker_encoder.encode_batch(wav.unsqueeze(0))
    return emb.squeeze().detach().cpu().numpy().astype(np.float32)


def cos_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])


# Tính target centroid (chỉ 1 lần)
target_eval_manifest = pd.read_csv(PREPARED_DIR / 'target_eval_manifest.csv')
source_manifest_df   = pd.read_csv(PREPARED_DIR / 'source_test_manifest.csv')

print('\nTính target embedding centroid...')
target_eval_paths = [Path(r) for r in target_eval_manifest['path'].tolist()]
target_centroid = np.mean(
    np.vstack([speaker_embedding(p) for p in tqdm(target_eval_paths, desc='Target emb')]),
    axis=0
)

print('Tính source embeddings baseline...')
source_embeddings: dict = {}
for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='Source emb'):
    source_embeddings[row['file']] = speaker_embedding(Path(row['path']))

print(f'✅ Target centroid + {len(source_embeddings)} source embeddings sẵn sàng')

# ---- ASR (Whisper) ----
from transformers import pipeline as hf_pipeline

print(f'\nLoad ASR: {ASR_MODEL_ID}...')
asr_pipe = hf_pipeline(
    'automatic-speech-recognition',
    model=ASR_MODEL_ID,
    device=0 if DEVICE == 'cuda' else -1,
)
print('✅ ASR OK')


def normalize_text(s: str) -> str:
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFC', s)
    s = re.sub(r'[^\w\s]', ' ', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def transcribe_audio(path: Path) -> str:
    kwargs = {}
    if ASR_LANGUAGE:
        kwargs['generate_kwargs'] = {'language': ASR_LANGUAGE, 'task': 'transcribe'}
    result = asr_pipe(str(path), **kwargs)
    return result.get('text', '') if isinstance(result, dict) else str(result)


# ---- MOS Predictor (UTMOS) ----
mos_predictor = None
mos_backend_available = False


def try_load_utmos():
    try:
        pred = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True)
        if hasattr(pred, 'to'):
            pred = pred.to(DEVICE)
        if hasattr(pred, 'eval'):
            pred.eval()
        return pred
    except Exception as e:
        print(f'UTMOS load thất bại: {e}')
        return None


def predict_mos_utmos(path: Path) -> float:
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.to(DEVICE)
    with torch.no_grad():
        for fn in [lambda: mos_predictor(wav, sr), lambda: mos_predictor(wav.squeeze(0), sr)]:
            try:
                score = fn()
                if isinstance(score, (tuple, list)):
                    score = score[0]
                if isinstance(score, torch.Tensor):
                    score = score.detach().cpu().numpy()
                return float(np.asarray(score).reshape(-1)[0])
            except Exception:
                continue
    raise RuntimeError(f'Không predict được MOS: {path}')


if RUN_PREDICTED_MOS and MOS_BACKEND == 'utmos_torchhub':
    print('\nLoad UTMOS...')
    mos_predictor = try_load_utmos()
    mos_backend_available = mos_predictor is not None
    print('✅ UTMOS OK' if mos_backend_available else '⚠️  UTMOS không khả dụng — MOS sẽ ghi NaN')

# ---- VC Inference Pipeline ----
print('\nLoad VC inference pipeline...')
_saved_argv = sys.argv
sys.argv = ['rvc_infer']
try:
    from configs.config import Config as RVCConfig
    rvc_config = RVCConfig()
finally:
    sys.argv = _saved_argv

from infer.modules.vc.modules import VC
from infer.modules.vc.utils import load_hubert

vc_pipeline = VC(rvc_config)
vc_pipeline.hubert_model = load_hubert(rvc_config)
print('✅ VC pipeline sẵn sàng')

In [ ]:
# ================================================================
# VÒNG LẶP CHÍNH: Training → Export → Inference → Đánh giá
# ================================================================
all_sim_rows: List[dict] = []
all_mos_rows: List[dict] = []
all_asr_rows: List[dict] = []

source_manifest_df = pd.read_csv(PREPARED_DIR / 'source_test_manifest.csv')

print(f'Sẽ chạy {len(EPOCH_LIST_EFFECTIVE)} model: {[f"model_{e}ep" for e in EPOCH_LIST_EFFECTIVE]}')
t_pipeline_start = time.time()

for epoch_target in EPOCH_LIST_EFFECTIVE:
    model_name = f'model_{epoch_target}ep'
    t_model_start = time.time()

    print(f'\n{"="*65}')
    print(f'  MODEL: {model_name}  |  target epoch: {epoch_target}')
    print(f'{"="*65}')

    # --------------------------------------------------------
    # 1. TRAINING
    # Training tự động CONTINUE từ checkpoint mới nhất nếu có.
    # Với EPOCH_LIST=[50,100,200]: lần 1 train 0→50, lần 2 50→100, lần 3 100→200.
    # --------------------------------------------------------
    print(f'\n[1/4] Training → epoch {epoch_target}...')
    print(f'      Log: logs/{EXPERIMENT_NAME}/train.log')
    p.total_epochs    = epoch_target
    p.infer_weight_name = model_name
    train_steps.step_train(STANDALONE_ROOT, config, p)

    # --------------------------------------------------------
    # 2. EXPORT MODEL NHỎ
    # --------------------------------------------------------
    print(f'\n[2/4] Export → assets/weights/{model_name}.pth')
    p.g_checkpoint_for_extract = ''  # rỗng = tự chọn checkpoint mới nhất
    export_result = train_steps.step_extract_small_weights(STANDALONE_ROOT, p)
    print(export_result)

    model_pth = STANDALONE_ROOT / 'assets' / 'weights' / f'{model_name}.pth'
    if not model_pth.exists():
        print(f'⚠️  {model_pth.name} không tìm thấy — bỏ qua inference + eval cho model này')
        continue
    print(f'  {model_pth.name}  ({model_pth.stat().st_size/1024**2:.1f} MB)')

    # --------------------------------------------------------
    # 3. INFERENCE
    # --------------------------------------------------------
    print(f'\n[3/4] Inference → {CONVERTED_ROOT / model_name}/')
    vc_pipeline.get_vc(f'{model_name}.pth')

    exp_dir = STANDALONE_ROOT / 'logs' / EXPERIMENT_NAME
    index_files = sorted(exp_dir.glob('added_*.index'), key=lambda f: f.stat().st_mtime)
    file_index = str(index_files[-1]) if (index_files and not SKIP_INDEX and INFER_INDEX_RATE > 0) else ''
    print(f'  Index: {Path(file_index).name if file_index else "(không dùng)"}')

    out_dir = CONVERTED_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc=f'  Infer {model_name}'):
        dst = out_dir / row['file']
        try:
            info, audio_out = vc_pipeline.vc_single(
                INFER_SPEAKER_ID, row['path'], INFER_F0_UP_KEY, None,
                INFER_F0_METHOD, file_index, '', INFER_INDEX_RATE,
                INFER_FILTER_RADIUS, INFER_RESAMPLE_SR, INFER_RMS_MIX_RATE, INFER_PROTECT,
            )
            if audio_out:
                tgt_sr, audio_i16 = audio_out
                sf.write(str(dst), audio_i16, tgt_sr)
        except Exception as e:
            print(f'  Lỗi infer {row["file"]}: {e}')

    n_converted = len(list(out_dir.glob('*.wav')))
    print(f'  {n_converted}/{len(source_manifest_df)} file đã convert')

    # --------------------------------------------------------
    # 4. ĐÁNH GIÁ
    # --------------------------------------------------------
    print(f'\n[4/4] Đánh giá {model_name}...')

    # --- Speaker Similarity ---
    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='  Speaker Sim'):
        converted = out_dir / row['file']
        src_tgt = cos_sim(source_embeddings[row['file']], target_centroid)
        if not converted.exists():
            all_sim_rows.append({
                'model': model_name, 'epoch': epoch_target, 'file': row['file'],
                'source_target_similarity': src_tgt,
                'converted_target_similarity': np.nan,
                'improvement': np.nan, 'converted_exists': False,
            })
            continue
        conv_emb = speaker_embedding(converted)
        conv_tgt = cos_sim(conv_emb, target_centroid)
        all_sim_rows.append({
            'model': model_name, 'epoch': epoch_target, 'file': row['file'],
            'source_target_similarity': src_tgt,
            'converted_target_similarity': conv_tgt,
            'improvement': conv_tgt - src_tgt, 'converted_exists': True,
        })

    # --- Predicted MOS ---
    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='  MOS'):
        converted = out_dir / row['file']
        base = {'model': model_name, 'epoch': epoch_target, 'file': row['file']}
        if not converted.exists():
            all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': 'missing converted'})
        elif not (RUN_PREDICTED_MOS and mos_backend_available):
            all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': 'MOS backend unavailable'})
        else:
            try:
                score = predict_mos_utmos(converted)
                all_mos_rows.append({**base, 'predicted_mos': score, 'mos_error': ''})
            except Exception as e:
                all_mos_rows.append({**base, 'predicted_mos': np.nan, 'mos_error': str(e)})

    # --- ASR WER/CER ---
    for _, row in tqdm(source_manifest_df.iterrows(), total=len(source_manifest_df), desc='  ASR'):
        converted = out_dir / row['file']
        ref = normalize_text(row['transcript'])
        base = {'model': model_name, 'epoch': epoch_target, 'file': row['file'], 'reference': ref}
        if not converted.exists():
            all_asr_rows.append({**base, 'hypothesis': '', 'wer': np.nan, 'cer': np.nan, 'asr_error': 'missing'})
        else:
            try:
                hyp = normalize_text(transcribe_audio(converted))
                all_asr_rows.append({
                    **base, 'hypothesis': hyp,
                    'wer': jiwer.wer(ref, hyp) if ref else np.nan,
                    'cer': jiwer.cer(ref, hyp) if ref else np.nan,
                    'asr_error': '',
                })
            except Exception as e:
                all_asr_rows.append({**base, 'hypothesis': '', 'wer': np.nan, 'cer': np.nan, 'asr_error': str(e)})

    elapsed = time.time() - t_model_start
    print(f'\n✅ {model_name} hoàn thành trong {elapsed/60:.0f} phút ({elapsed/3600:.1f} giờ)')

# ---- Lưu tất cả kết quả ----
print('\nLưu kết quả...')
sim_df = pd.DataFrame(all_sim_rows)
mos_df = pd.DataFrame(all_mos_rows)
asr_df = pd.DataFrame(all_asr_rows)

for df, name in [(sim_df, 'speaker_similarity'), (mos_df, 'predicted_mos'), (asr_df, 'asr_wer_cer')]:
    if not df.empty:
        csv_path = RESULTS_DIR / f'{name}_results.csv'
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f'  Lưu: {csv_path}')

total_elapsed = time.time() - t_pipeline_start
print(f'\n✅ VÒNG LẶP CHÍNH HOÀN THÀNH trong {total_elapsed/3600:.1f} giờ')

In [ ]:
# ================================================================
# TỔNG HỢP KẾT QUẢ + BẢNG SO SÁNH + BIỂU ĐỒ
# (Cell này có thể chạy độc lập sau khi vòng lặp xong)
# ================================================================

def load_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


sim_df = load_csv_if_exists(RESULTS_DIR / 'speaker_similarity_results.csv')
mos_df = load_csv_if_exists(RESULTS_DIR / 'predicted_mos_results.csv')
asr_df = load_csv_if_exists(RESULTS_DIR / 'asr_wer_cer_results.csv')

# Merge per-sample
per_sample = sim_df.copy() if not sim_df.empty else None
if per_sample is not None and not mos_df.empty:
    per_sample = per_sample.merge(
        mos_df[['model', 'file', 'predicted_mos', 'mos_error']],
        on=['model', 'file'], how='left'
    )
if per_sample is not None and not asr_df.empty:
    per_sample = per_sample.merge(
        asr_df[['model', 'file', 'reference', 'hypothesis', 'wer', 'cer', 'asr_error']],
        on=['model', 'file'], how='left'
    )

if per_sample is None or per_sample.empty:
    print('Chưa có kết quả. Chạy Cell 10 trước.')
else:
    per_sample.to_csv(RESULTS_DIR / 'per_sample_results.csv', index=False, encoding='utf-8-sig')

    # Bảng tổng hợp
    agg_dict = {
        'source_target_similarity':   'mean',
        'converted_target_similarity':'mean',
        'improvement':                'mean',
    }
    group_cols = ['model']
    if 'epoch' in per_sample.columns:
        group_cols = ['model', 'epoch']
    if 'predicted_mos' in per_sample.columns:
        agg_dict['predicted_mos'] = 'mean'
    if 'wer' in per_sample.columns:
        agg_dict['wer'] = 'mean'
    if 'cer' in per_sample.columns:
        agg_dict['cer'] = 'mean'

    summary = per_sample.groupby(group_cols).agg(agg_dict).reset_index()
    summary['n_samples'] = per_sample.groupby(group_cols)['file'].count().values
    if 'epoch' in summary.columns:
        summary = summary.sort_values('epoch')

    summary.to_csv(RESULTS_DIR / 'summary_by_model.csv', index=False, encoding='utf-8-sig')

    print('BẢNG TỔNG HỢP:')
    try:
        display(summary)
    except NameError:
        print(summary.to_string())

    # Biểu đồ
    metrics_cfg = [
        ('converted_target_similarity', 'Speaker Similarity ↑'),
        ('predicted_mos',               'Predicted MOS ↑'),
        ('wer',                         'WER ↓'),
        ('cer',                         'CER ↓'),
    ]
    active = [(m, t) for m, t in metrics_cfg if m in summary.columns and not summary[m].isna().all()]

    if active:
        fig, axes = plt.subplots(1, len(active), figsize=(4 * len(active), 4), squeeze=False)
        x_labels = summary['model'].tolist()

        for ax_i, (metric, title) in enumerate(active):
            ax = axes[0][ax_i]
            ax.bar(x_labels, summary[metric], color='steelblue', edgecolor='white')
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Model')
            ax.set_ylabel(metric)
            ax.tick_params(axis='x', rotation=30)
            for bar, val in zip(ax.patches, summary[metric]):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        out_plot = RESULTS_DIR / 'summary_plot.png'
        plt.savefig(out_plot, dpi=160, bbox_inches='tight')
        plt.show()
        print(f'Biểu đồ lưu tại: {out_plot}')

    print(f'\nFile kết quả:')
    for csv_name in ['per_sample_results.csv', 'summary_by_model.csv']:
        p_csv = RESULTS_DIR / csv_name
        if p_csv.exists():
            print(f'  ✅ {p_csv}')

## Cách đọc kết quả

File `results/summary_by_model.csv` là bảng chính để so sánh.

| Cột | Ý nghĩa | Hướng tốt |
|-----|---------|----------|
| `converted_target_similarity` | Độ giống giọng converted với giọng target | ↑ Càng cao càng tốt |
| `improvement` | Tăng similarity so với giọng nguồn ban đầu | ↑ Càng cao càng tốt |
| `predicted_mos` | Điểm chất lượng âm thanh (1–5) | ↑ Càng cao càng tốt |
| `wer` | Word Error Rate — nội dung lời nói bị sai bao nhiêu % | ↓ Càng thấp càng tốt |
| `cer` | Character Error Rate | ↓ Càng thấp càng tốt |

## Lưu ý về vòng lặp training

Với `EPOCH_LIST = [50, 100, 200]`:
- Lần 1: train từ **0 → 50** epoch → export `model_50ep.pth`
- Lần 2: **tiếp tục** từ checkpoint 50 → **100** epoch → export `model_100ep.pth`  
- Lần 3: **tiếp tục** từ checkpoint 100 → **200** epoch → export `model_200ep.pth`

Training không bao giờ bắt đầu lại từ đầu — luôn tiếp tục từ checkpoint cuối.

## Điều chỉnh nếu kết quả kém

| Vấn đề | Cách khắc phục |
|--------|----------------|
| Similarity thấp | Train thêm epoch, thêm data target |
| WER/CER cao | Giảm `INFER_INDEX_RATE`, kiểm tra `INFER_F0_UP_KEY` |
| MOS thấp | Tăng `INFER_PROTECT`, giảm `INFER_RMS_MIX_RATE` |
| Giọng sai tông | Điều chỉnh `INFER_F0_UP_KEY` ±5 đến ±7 nếu khác giới tính |